## Run Optuna Hyper-parameter Optimizer for Building Detection Modeling using Deep Learning.

In [ ]:
import gc
from typing import Tuple

import optuna
import torch
from arcgis.learn import UnetClassifier, prepare_data
from fastai.vision.transform import brightness, contrast, crop, rand_zoom, rotate, skew

### Check if GPU is enabled.

In [ ]:
torch.cuda.is_available()

In [ ]:
class_name = "Building"
data_path = "ETD4DL352"

### Define standalone fit method.

In [ ]:
def fit(
    backbone: str,
    chip_size: int,
    epochs: int = 100,
    checkpoint=True,
    compute_metrics=True,
) -> Tuple[float, float]:
    """
    Perform full model fit with defined parameters.

    :param backbone: The model backbone.
    :param chip_size: The transformation chip size.
    :param epochs: The number of epochs per fit.
    :param checkpoint: Save checkpoints.
    :param compute_metrics: Save model at the end of epochs.
    :return: (precision,recall)
    """

    # get a batch size given a backbone and a chip size.
    batch_size = {
        "resnet18:224": 16,
        "resnet18:256": 12,
        "resnet18:288": 10,
        "resnet18:320": 8,
        "resnet18:352": 6,
        "resnet34:224": 12,
        "resnet34:256": 10,
        "resnet34:288": 8,
        "resnet34:320": 8,
        "resnet34:352": 6,
    }[f"{backbone}:{chip_size}"]

    print(backbone, chip_size, batch_size)

    # Create training transformation.
    ranges = (0, 1)
    transform_train = [
        crop(size=chip_size, p=1.0, row_pct=ranges, col_pct=ranges),
        brightness(change=(0.4, 0.6)),
        contrast(scale=(0.75, 1.5)),
        rand_zoom(scale=(0.90, 1.10)),
    ]

    # Create validation transformation.
    transform_valid = [crop(size=chip_size, p=1.0, row_pct=0.5, col_pct=0.5)]
    transforms = (transform_train, transform_valid)
    # Read the data.
    data = prepare_data(
        data_path, chip_size=chip_size, batch_size=batch_size, transforms=transforms
    )
    # Select a head and a backbone.
    unet = UnetClassifier(
        data,
        backbone=backbone,
        mixup=True,
        checkpoint=checkpoint,
    )
    unet.unfreeze()
    # Fit the model.
    unet.fit(epochs, lr=1e-4, early_stopping=True)
    # Save the model.
    if compute_metrics:
        unet.save(f"unet_{backbone}_{chip_size}", compute_metrics=compute_metrics)
    df = unet.per_class_metrics()
    # Get the recall and precision.
    precision = df[class_name]["precision"]
    recall = df[class_name]["recall"]
    # Clear the memory.
    del unet
    del data
    torch.cuda.empty_cache()
    gc.collect()
    return precision, recall

### Following cell is to perform a "simple" fit test.

In [ ]:
# fit("resnet18", chip_size=352, epochs=2, checkpoint=False, compute_metrics=False)

### Define the optuna objective function.

In [ ]:
def objective(trial) -> Tuple[float, float]:
    # Suggest a chip size.
    chip_size = trial.suggest_int("chip_size", 352 - 4 * 32, 352, 32)
    # Suggest a backbone.
    backbone = trial.suggest_categorical("backbone", ["resnet18", "resnet34"])
    # Run fit and return precision and recall.
    return fit(backbone, chip_size)

### Define a study where we are optimizing the precision and recall.

Here we are forcing a GC to avoid an OOM Exception.

In [ ]:
study = optuna.create_study(directions=["maximize", "maximize"])
study.optimize(objective, n_trials=20, gc_after_trial=True)

### Report the best results.

In [ ]:
print(f"Number of trials : {len(study.best_trials)}")

trial_with_highest_recall = max(study.best_trials, key=lambda t: t.values[1])

print(f"Trial with highest precision, recall: ")
print(f"\tnumber: {trial_with_highest_recall.number}")
print(f"\tparams: {trial_with_highest_recall.params}")
print(f"\tvalues: {trial_with_highest_recall.values}")